In [ ]:
# Домашнее задание HW10-11: компьютерное зрение в PyTorch

In [5]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 -i https://pypi.douban.com/simple

Looking in indexes: https://pypi.douban.com/simple


In [6]:
import torch
print("Версия PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Имя устройства:", torch.cuda.get_device_name(0))

Версия PyTorch: 2.7.1+cu118
CUDA доступна: True
Имя устройства: NVIDIA GeForce RTX 3060 Ti


In [7]:
import os
import random
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms, models
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import nms
from PIL import Image
import pandas as pd

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

Используется устройство: cuda


In [8]:
PROJECT_ROOT = os.getcwd()
HW_ROOT = PROJECT_ROOT
ARTIFACTS_DIR = os.path.join(HW_ROOT, "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

DATA_DIR = os.path.join(HW_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Корень HW: {HW_ROOT}")
print(f"Данные будут в: {DATA_DIR}")
print(f"Артефакты: {ARTIFACTS_DIR}")

Корень HW: C:\Users\Trixx\short-aie\short-and-memorable-aie\homeworks\HW10-11
Данные будут в: C:\Users\Trixx\short-aie\short-and-memorable-aie\homeworks\HW10-11\data
Артефакты: C:\Users\Trixx\short-aie\short-and-memorable-aie\homeworks\HW10-11\artifacts


In [ ]:
## Часть A: классификация STL10
# Трансформы для STL10
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Базовый трансформ
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Аугментации C2
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

resnet_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train = datasets.STL10(root=DATA_DIR, split='train', download=True)
test_dataset = datasets.STL10(root=DATA_DIR, split='test', download=True)

train_size = int(0.8 * len(full_train))
val_size = len(full_train) - train_size
train_data_raw, val_data_raw = random_split(full_train, [train_size, val_size])

class TransformWrapper:
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return self.transform(img), label

# C1: simple-cnn-base
train_c1 = TransformWrapper(train_data_raw, base_transform)
val_c1 = TransformWrapper(val_data_raw, base_transform)
test_c1 = TransformWrapper(test_dataset, base_transform)

# C2: simple-cnn-aug
train_c2 = TransformWrapper(train_data_raw, aug_transform)
val_c2 = TransformWrapper(val_data_raw, base_transform)  # валидация без аугментаций
test_c2 = TransformWrapper(test_dataset, base_transform)

# C3 и C4
train_c3 = TransformWrapper(train_data_raw, resnet_preprocess)
val_c3 = TransformWrapper(val_data_raw, resnet_preprocess)
test_c3 = TransformWrapper(test_dataset, resnet_preprocess)

train_c4 = train_c3
val_c4 = val_c3
test_c4 = test_c3

batch_size_cnn = 64
batch_size_resnet = 32

dataloaders = {
    'C1_train': DataLoader(train_c1, batch_size_cnn, shuffle=True),
    'C1_val': DataLoader(val_c1, batch_size_cnn, shuffle=False),
    'C2_train': DataLoader(train_c2, batch_size_cnn, shuffle=True),
    'C2_val': DataLoader(val_c2, batch_size_cnn, shuffle=False),
    'C3_train': DataLoader(train_c3, batch_size_resnet, shuffle=True),
    'C3_val': DataLoader(val_c3, batch_size_resnet, shuffle=False),
    'C4_train': DataLoader(train_c4, batch_size_resnet, shuffle=True),
    'C4_val': DataLoader(val_c4, batch_size_resnet, shuffle=False),
}
test_loader_cnn = DataLoader(test_c1, batch_size_cnn, shuffle=False)
test_loader_resnet = DataLoader(test_c3, batch_size_resnet, shuffle=False)

# Sanity check
for name, loader in list(dataloaders.items())[:2]:
    images, labels = next(iter(loader))
    print(f"{name}: x.shape = {images.shape}, y.shape = {labels.shape}")

In [ ]:
def show_images(images, labels, class_names=None, title=""):
    fig, axes = plt.subplots(1, min(5, len(images)), figsize=(12,3))
    for i, ax in enumerate(axes):
        img = images[i].permute(1,2,0).numpy()
        img = img * std + mean 
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"Label: {labels[i].item()}")
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "augmentations_preview.png"), dpi=150)
    plt.show()

sample_batch, _ = next(iter(dataloaders['C2_train']))
show_images(sample_batch, _, title="Аугментированные примеры (C2)")
sample_batch_base, _ = next(iter(dataloaders['C1_train']))
show_images(sample_batch_base, _, title="Базовые примеры (C1)")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # После трёх пулингов: 96x96 -> 12x12
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item() * x.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

def run_experiment(experiment_id, model, train_loader, val_loader, epochs, lr, device, patience=5):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    counter = 0
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f"{experiment_id} Epoch {epoch+1}/{epochs}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    return best_model_state, best_val_acc, history

# Эксперименты C1 и C2
epochs_cnn = 15
lr_cnn = 1e-3

model_c1 = SimpleCNN(num_classes=10)
best_c1, best_val_c1, hist_c1 = run_experiment("C1", model_c1, dataloaders['C1_train'], dataloaders['C1_val'], epochs_cnn, lr_cnn, device)

model_c2 = SimpleCNN(num_classes=10)
best_c2, best_val_c2, hist_c2 = run_experiment("C2", model_c2, dataloaders['C2_train'], dataloaders['C2_val'], epochs_cnn, lr_cnn, device)

In [ ]:
# C3: ResNet18 head-only
model_c3 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
# Заморозка backbone
for param in model_c3.parameters():
    param.requires_grad = False
# Замена fc
num_classes = 10
model_c3.fc = nn.Linear(model_c3.fc.in_features, num_classes)
model_c3.to(device)

# Оптимизатор только для fc
optimizer_c3 = optim.Adam(model_c3.fc.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

def train_head_only(model, train_loader, val_loader, optimizer, criterion, device, epochs=15):
    model.train()
    best_val_acc = 0
    best_state = None
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        # train
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == y).sum().item()
            train_total += y.size(0)
        train_loss /= train_total
        train_acc = train_correct / train_total
        # val
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs, y)
                val_loss += loss.item() * x.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
        val_loss /= val_total
        val_acc = val_correct / val_total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f"C3 Epoch {epoch+1}/{epochs}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict().copy()
    return best_state, best_val_acc, history

best_c3, best_val_c3, hist_c3 = train_head_only(model_c3, dataloaders['C3_train'], dataloaders['C3_val'], optimizer_c3, criterion, device, epochs=15)

In [ ]:
model_c4 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for name, param in model_c4.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

model_c4.fc = nn.Linear(model_c4.fc.in_features, num_classes)
model_c4.to(device)

optimizer_c4 = optim.Adam(filter(lambda p: p.requires_grad, model_c4.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()

best_c4, best_val_c4, hist_c4 = train_head_only(model_c4, dataloaders['C4_train'], dataloaders['C4_val'], optimizer_c4, criterion, device, epochs=15)

In [ ]:
results_c = {
    'C1': best_val_c1,
    'C2': best_val_c2,
    'C3': best_val_c3,
    'C4': best_val_c4
}
best_exp = max(results_c, key=results_c.get)
print(f"Лучший эксперимент по val_accuracy: {best_exp} с accuracy = {results_c[best_exp]:.4f}")

if best_exp == 'C1':
    best_model_state = best_c1
    best_model_arch = 'SimpleCNN'
elif best_exp == 'C2':
    best_model_state = best_c2
    best_model_arch = 'SimpleCNN'
elif best_exp == 'C3':
    best_model_state = best_c3
    best_model_arch = 'ResNet18_head_only'
else:
    best_model_state = best_c4
    best_model_arch = 'ResNet18_finetune'

torch.save(best_model_state, os.path.join(ARTIFACTS_DIR, 'best_classifier.pt'))
print(f"Модель сохранена: {os.path.join(ARTIFACTS_DIR, 'best_classifier.pt')}")

if best_exp in ['C1','C2']:
    test_loader = test_loader_cnn
else:
    test_loader = test_loader_resnet

if best_exp in ['C1','C2']:
    test_model = SimpleCNN(num_classes=10)
else:
    test_model = models.resnet18(weights=None)
    if best_exp == 'C3':
        test_model.fc = nn.Linear(test_model.fc.in_features, 10)
    else:
        for name, param in test_model.named_parameters():
            param.requires_grad = True
        test_model.fc = nn.Linear(test_model.fc.in_features, 10)
test_model.load_state_dict(best_model_state)
test_model.to(device)
test_loss, test_acc = evaluate(test_model, test_loader, nn.CrossEntropyLoss(), device)
print(f"Test accuracy лучшей модели: {test_acc:.4f}")

In [ ]:
# Визуализация результатов C1-C4
exp_names = list(results_c.keys())
val_accs = list(results_c.values())
plt.figure(figsize=(8,5))
plt.bar(exp_names, val_accs, color=['blue','green','orange','red'])
plt.ylim(0,1)
plt.ylabel('Validation Accuracy')
plt.title('Сравнение экспериментов C1-C4')
for i, v in enumerate(val_accs):
    plt.text(i, v+0.01, f"{v:.3f}", ha='center')
plt.savefig(os.path.join(FIGURES_DIR, 'classification_compare.png'), dpi=150)
plt.show()

hist_best = hist_c4 if best_exp=='C4' else (hist_c3 if best_exp=='C3' else (hist_c2 if best_exp=='C2' else hist_c1))
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist_best['train_loss'], label='train')
plt.plot(hist_best['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title(f'{best_exp} Loss')
plt.subplot(1,2,2)
plt.plot(hist_best['train_acc'], label='train')
plt.plot(hist_best['val_acc'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title(f'{best_exp} Accuracy')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'classification_curves_best.png'), dpi=150)
plt.show()

In [ ]:
## Часть B: детекция Pascal VOC
from torchvision.datasets import VOCDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
import torchvision.transforms as T

class VOCTransform:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        image = F.normalize(image, mean=mean, std=std)
        boxes = torch.as_tensor(target['annotation']['object'], dtype=torch.float32)
        bbox_list = []
        labels_list = []
        for obj in target['annotation']['object']:
            bndbox = obj['bndbox']
            xmin = float(bndbox['xmin'])
            ymin = float(bndbox['ymin'])
            xmax = float(bndbox['xmax'])
            ymax = float(bndbox['ymax'])
            bbox_list.append([xmin, ymin, xmax, ymax])
            label = int(obj['name'])  # в VOC метки от 1 до 20, но у нас будут индексы классов
            labels_list.append(label)
        if len(bbox_list) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(bbox_list, dtype=torch.float32)
            labels = torch.as_tensor(labels_list, dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels}
        return image, target

voc_val = VOCDetection(root=os.path.join(DATA_DIR, 'VOC'), year='2012', image_set='val', download=True, transform=None)
class VOCDetectionWrapper:
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, target = self.dataset[idx]
        return self.transform(img, target)

val_dataset = VOCDetectionWrapper(voc_val, VOCTransform())
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

model_det = fasterrcnn_resnet50_fpn(weights='DEFAULT')
model_det.eval()
model_det.to(device)


def predict_with_threshold(model, images, score_threshold=0.5, iou_thresh=0.5):
    model.eval()
    with torch.no_grad():
        outputs = model(images)
    results = []
    for out in outputs:
        boxes = out['boxes'].cpu()
        scores = out['scores'].cpu()
        labels = out['labels'].cpu()
        keep = scores > score_threshold
        boxes = boxes[keep]
        scores = scores[keep]
        labels = labels[keep]
        if len(boxes) > 0:
            keep_nms = nms(boxes, scores, iou_thresh)
            boxes = boxes[keep_nms]
            scores = scores[keep_nms]
            labels = labels[keep_nms]
        results.append({'boxes': boxes, 'scores': scores, 'labels': labels})
    return results

In [ ]:
def compute_metrics(predictions, targets, iou_threshold=0.5):
    tp = 0
    fp = 0
    fn = 0
    total_iou = 0
    matched_preds = 0
    for pred, target in zip(predictions, targets):
        gt_boxes = target['boxes']
        pred_boxes = pred['boxes']
        if len(gt_boxes) == 0:
            fp += len(pred_boxes)
            continue
        if len(pred_boxes) == 0:
            fn += len(gt_boxes)
            continue
        matched_gt = set()
        for i, pbox in enumerate(pred_boxes):
            best_iou = 0
            best_gt_idx = -1
            for j, gbox in enumerate(gt_boxes):
                iou = box_iou(pbox.unsqueeze(0), gbox.unsqueeze(0)).item()
                if iou > best_iou and j not in matched_gt:
                    best_iou = iou
                    best_gt_idx = j
            if best_iou >= iou_threshold and best_gt_idx != -1:
                tp += 1
                matched_gt.add(best_gt_idx)
                total_iou += best_iou
                matched_preds += 1
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp+fp)>0 else 0
    recall = tp / (tp + fn) if (tp+fn)>0 else 0
    mean_iou = total_iou / matched_preds if matched_preds>0 else 0
    return precision, recall, mean_iou

def box_iou(box1, box2):
    # box1: (1,4), box2: (1,4)
    x1 = max(box1[0,0], box2[0,0])
    y1 = max(box1[0,1], box2[0,1])
    x2 = min(box1[0,2], box2[0,2])
    y2 = min(box1[0,3], box2[0,3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[0,2]-box1[0,0]) * (box1[0,3]-box1[0,1])
    area2 = (box2[0,2]-box2[0,0]) * (box2[0,3]-box2[0,1])
    union = area1 + area2 - inter
    return inter / union if union>0 else 0

In [ ]:
thresholds = [0.3, 0.7]
metrics_results = {}
for thresh in thresholds:
    all_preds = []
    all_targets = []
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        preds = predict_with_threshold(model_det, images, score_threshold=thresh)
        all_preds.extend(preds)
        all_targets.extend(targets)
    prec, rec, miou = compute_metrics(all_preds, all_targets, iou_threshold=0.5)
    metrics_results[thresh] = {'precision': prec, 'recall': rec, 'mean_iou': miou}
    print(f"Threshold {thresh}: Precision={prec:.3f}, Recall={rec:.3f}, mIoU={miou:.3f}")

In [ ]:
def visualize_detections(images, targets, predictions, title, num_samples=3):
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1,-1)
    for i in range(num_samples):
        img = images[i].cpu().permute(1,2,0).numpy()
        img = img * std + mean
        img = np.clip(img, 0, 1)
        # Ground truth
        ax_gt = axes[i,0]
        ax_gt.imshow(img)
        ax_gt.set_title('Ground Truth')
        for box in targets[i]['boxes']:
            rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, edgecolor='green', linewidth=2)
            ax_gt.add_patch(rect)
        ax_gt.axis('off')
        # Predictions
        ax_pred = axes[i,1]
        ax_pred.imshow(img)
        ax_pred.set_title(f'Predictions (score>={title})')
        for box, score in zip(predictions[i]['boxes'], predictions[i]['scores']):
            if score > 0.5:  # показать только уверенные
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, edgecolor='red', linewidth=2)
                ax_pred.add_patch(rect)
                ax_pred.text(box[0], box[1]-5, f'{score:.2f}', color='red', fontsize=8)
        ax_pred.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'detection_examples_{title}.png'), dpi=150)
    plt.show()

sample_images, sample_targets = next(iter(val_loader))
sample_images_gpu = [img.to(device) for img in sample_images]
preds_03 = predict_with_threshold(model_det, sample_images_gpu, score_threshold=0.3)
preds_07 = predict_with_threshold(model_det, sample_images_gpu, score_threshold=0.7)
visualize_detections(sample_images, sample_targets, preds_03, title="0.3", num_samples=min(3, len(sample_images)))
visualize_detections(sample_images, sample_targets, preds_07, title="0.7", num_samples=min(3, len(sample_images)))

In [ ]:
thresh_list = list(metrics_results.keys())
prec_list = [metrics_results[t]['precision'] for t in thresh_list]
rec_list = [metrics_results[t]['recall'] for t in thresh_list]
miou_list = [metrics_results[t]['mean_iou'] for t in thresh_list]

plt.figure(figsize=(8,5))
x = np.arange(len(thresh_list))
width = 0.25
plt.bar(x - width, prec_list, width, label='Precision')
plt.bar(x, rec_list, width, label='Recall')
plt.bar(x + width, miou_list, width, label='mIoU')
plt.xticks(x, [f"Thresh {t}" for t in thresh_list])
plt.ylabel('Score')
plt.title('Detection Metrics for V1 and V2')
plt.legend()
plt.savefig(os.path.join(FIGURES_DIR, 'detection_metrics.png'), dpi=150)
plt.show()

In [ ]:
runs_data = [
    {'experiment_id': 'C1', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': 15, 'best_val_accuracy': best_val_c1, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'no augmentations'},
    {'experiment_id': 'C2', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'SimpleCNN', 'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': 15, 'best_val_accuracy': best_val_c2, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'with augmentations'},
    {'experiment_id': 'C3', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'ResNet18_head_only', 'optimizer': 'Adam', 'lr': 1e-4, 'epochs_trained': 15, 'best_val_accuracy': best_val_c3, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'pretrained backbone frozen'},
    {'experiment_id': 'C4', 'task': 'classification', 'dataset': 'STL10', 'seed': 42, 'model_summary': 'ResNet18_finetune', 'optimizer': 'Adam', 'lr': 1e-4, 'epochs_trained': 15, 'best_val_accuracy': best_val_c4, 'test_accuracy': None, 'precision': None, 'recall': None, 'mean_iou': None, 'notes': 'layer4+fc unfrozen'},
    {'experiment_id': 'V1', 'task': 'detection', 'dataset': 'PascalVOC', 'seed': 42, 'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': None, 'lr': None, 'epochs_trained': 0, 'best_val_accuracy': None, 'test_accuracy': None, 'precision': metrics_results[0.3]['precision'], 'recall': metrics_results[0.3]['recall'], 'mean_iou': metrics_results[0.3]['mean_iou'], 'notes': 'score_threshold=0.3'},
    {'experiment_id': 'V2', 'task': 'detection', 'dataset': 'PascalVOC', 'seed': 42, 'model_summary': 'FasterRCNN_ResNet50_FPN', 'optimizer': None, 'lr': None, 'epochs_trained': 0, 'best_val_accuracy': None, 'test_accuracy': None, 'precision': metrics_results[0.7]['precision'], 'recall': metrics_results[0.7]['recall'], 'mean_iou': metrics_results[0.7]['mean_iou'], 'notes': 'score_threshold=0.7'}
]
for row in runs_data:
    if row['experiment_id'] == best_exp:
        row['test_accuracy'] = test_acc
    elif row['task']=='classification':
        row['test_accuracy'] = None

runs_df = pd.DataFrame(runs_data)
runs_df.to_csv(os.path.join(ARTIFACTS_DIR, 'runs.csv'), index=False)
print(f"runs.csv сохранён в {ARTIFACTS_DIR}")

In [ ]:
best_config = {
    'dataset': 'STL10',
    'architecture': best_model_arch,
    'transforms': 'resize_norm' if best_exp in ['C3','C4'] else ('aug' if best_exp=='C2' else 'base'),
    'seed': 42,
    'optimizer': 'Adam',
    'learning_rate': 1e-4 if best_exp in ['C3','C4'] else 1e-3,
    'batch_size': batch_size_resnet if best_exp in ['C3','C4'] else batch_size_cnn,
    'epochs': 15,
    'best_val_accuracy': float(results_c[best_exp]),
    'test_accuracy': float(test_acc)
}
with open(os.path.join(ARTIFACTS_DIR, 'best_classifier_config.json'), 'w') as f:
    json.dump(best_config, f, indent=4)
print("Конфиг сохранён")